# Week 73, Day 5: Dependency management

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w073d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
index = {
    "web":    {"2.0": ["http>=2"], "1.0": ["http>=1"]},
    "report": {"1.0": ["http<2"]},
    "http":   {"2.1": [], "2.0": [], "1.5": []},
}

def newest_matching(name, low=None, high=None):
    versions = sorted(index[name], key=float, reverse=True)
    return next(v for v in versions if (low is None or float(v) >= low) and (high is None or float(v) < high))

print("web wants http", newest_matching("http", low=2.0))
print("report wants http", newest_matching("http", high=2.0))
print("so newest web can't be used with report; older web is", newest_matching("web", high=2.0), "with http", newest_matching("http", low=1.0, high=2.0))

In [ ]:
index = {
    "app":    {"1.0": {"web": ">=1", "report": ">=1"}},
    "web":    {"2.0": {"http": ">=2"}, "1.0": {"http": ">=1"}},
    "report": {"1.0": {"http": "<2"}},
    "http":   {"2.1": {}, "2.0": {}, "1.5": {}},
}

def ok(version, spec):
    op, number = spec[:2] if spec[1] == "=" else spec[:1], spec.lstrip("<>=")
    v, n = float(version), float(number)
    return {">=": v >= n, "<": v < n, ">": v > n, "<=": v <= n}[op]

def resolve(todo, chosen, constraints):
    if not todo:
        return chosen
    name, rest = todo[0], todo[1:]
    if name in chosen:
        return resolve(rest, chosen, constraints)
    for version in sorted(index[name], key=float, reverse=True):
        if all(ok(version, spec) for spec in constraints.get(name, [])):
            new_constraints = {k: list(v) for k, v in constraints.items()}
            for dep, spec in index[name][version].items():
                if dep in chosen and not ok(chosen[dep], spec):
                    break
                new_constraints.setdefault(dep, []).append(spec)
            else:
                print("try", name, version)
                result = resolve(rest + list(index[name][version]), {**chosen, name: version}, new_constraints)
                if result:
                    return result
                print("backtrack from", name, version)
    return None

print(resolve(["app"], {}, {}))

> **Key idea.** A specifier's clauses must all hold: ==, != (both with .* prefixes), <, <=, >, >=, and ~= (compatible release: ~=1.4.2 is >=1.4.2, ==1.4.*). Libraries declare wide ranges; applications also commit a lock file of every exact version, transitive ones included, with hashes. Development tools go in an extra or dependency groups. Resolvers search newest first, collect each choice's requirements, and backtrack when something can't be satisfied; a conflict means no combination exists. Upgrade regularly with tests running, audit for vulnerabilities, read changelogs, and keep dependencies few.

### Exercise: Matching specifiers

Write `version_tuple(v)` (whole numbers only, trailing zeros removed) and `matches(version, specifier)`: does `version` (such as `"1.4.2"`) satisfy every clause of `specifier` (such as `">=1.4, !=1.4.1, <2"`)? Support `==`, `!=`, `<`, `<=`, `>`, `>=` and `~=`, plus a `.*` suffix on `==` and `!=` meaning a prefix match on the numbers. `~=X.Y.Z` means `>=X.Y.Z, ==X.Y.*`; `~=` needs at least two numbers, otherwise raise `ValueError`. An empty specifier matches everything.

In [ ]:
def version_tuple(v):
    ___

def matches(version, specifier):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_matches():\n    \"every operator\"\n    assert version_tuple(\"1.4.0\") == (1, 4) and version_tuple(\"2\") == (2,)\n    assert matches(\"1.4.2\", \">=1.4, !=1.4.1, <2\") and not matches(\"1.4.1\", \">=1.4, !=1.4.1, <2\")\n    assert matches(\"1.4\", \"==1.4.0\") and matches(\"1.4.9\", \"==1.4.*\") and not matches(\"1.5\", \"==1.4.*\")\n    assert not matches(\"1.4.3\", \"!=1.4.*\") and matches(\"1.10\", \"!=1.1.*\")\n    assert matches(\"2.3\", \"~=2.3\") and matches(\"2.99\", \"~=2.3\") and not matches(\"3.0\", \"~=2.3\")\n    assert matches(\"1.4.5\", \"~=1.4.2\") and not matches(\"1.5.0\", \"~=1.4.2\") and not matches(\"1.4.1\", \"~=1.4.2\")\n    assert matches(\"0.1\", \"\") and matches(\"1.10\", \">1.9\")\n    try:\n        matches(\"1\", \"~=1\")\n        assert False\n    except ValueError:\n        pass")

### Exercise: A dependency tree

Write `tree(resolution, requires, root)`: `resolution` maps each package to its chosen version; `requires` maps each package to the list of packages it depends on. Return the lines of an indented tree from `root`, each `"name==version"`, indented two spaces per level, children in the order listed. A package that's already been shown is shown again as `"name==version (see above)"` without its children. Then `transitive(requires, root)`: the sorted list of every package `root` depends on, directly or not (excluding itself).

In [ ]:
def tree(resolution, requires, root):
    ___

def transitive(requires, root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tree():\n    \"direct and transitive dependencies\"\n    resolution = {\"app\": \"1.0\", \"web\": \"1.0\", \"report\": \"1.0\", \"http\": \"1.5\", \"idna\": \"3.7\"}\n    requires = {\"app\": [\"web\", \"report\"], \"web\": [\"http\"], \"report\": [\"http\", \"idna\"], \"http\": [\"idna\"], \"idna\": []}\n    assert tree(resolution, requires, \"app\") == [\n        \"app==1.0\", \"  web==1.0\", \"    http==1.5\", \"      idna==3.7\",\n        \"  report==1.0\", \"    http==1.5 (see above)\", \"    idna==3.7 (see above)\",\n    ]\n    assert transitive(requires, \"app\") == [\"http\", \"idna\", \"report\", \"web\"] and transitive(requires, \"idna\") == []")

### Exercise: A lock file with hashes

Write `lock_file(resolution, hashes)`: the text of a `requirements.txt` lock file for a resolution (a dict of name to version), in the format pip-tools writes: packages sorted by name, each as `name==version \` followed by one line per hash, `    --hash=sha256:HASH`, with a ` \` after every hash line except the last for that package; the file ends with a newline. `hashes` maps `(name, version)` to a list of hashes, written sorted. A missing entry raises `KeyError`. Then `parse_lock(text)`: back to a dict of name to `(version, sorted list of hashes)`.

In [ ]:
def lock_file(resolution, hashes):
    ___

def parse_lock(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lock():\n    \"pinned and hashed\"\n    resolution = {\"rich\": \"13.7.1\", \"idna\": \"3.7\"}\n    hashes = {(\"rich\", \"13.7.1\"): [\"bbb\", \"aaa\"], (\"idna\", \"3.7\"): [\"ccc\"]}\n    text = lock_file(resolution, hashes)\n    assert text == \"idna==3.7 \\\\\\n    --hash=sha256:ccc\\nrich==13.7.1 \\\\\\n    --hash=sha256:aaa \\\\\\n    --hash=sha256:bbb\\n\"\n    assert parse_lock(text) == {\"idna\": (\"3.7\", [\"ccc\"]), \"rich\": (\"13.7.1\", [\"aaa\", \"bbb\"])}\n    try:\n        lock_file({\"x\": \"1\"}, {})\n        assert False\n    except KeyError:\n        pass")

### Exercise: What's out of date?

Write `outdated(locked, available)`: `locked` maps names to the locked version; `available` maps names to lists of published versions (whole-number versions, in any order, possibly including the locked one). For each locked package whose newest available version is newer, report `(name, locked, newest, kind)`, where `kind` is `"major"`, `"minor"` or `"patch"`: the first of the three numbers that differs (pad versions to three numbers). Return the list sorted by kind (major first) and then by name.

In [ ]:
def outdated(locked, available):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_outdated():\n    \"an upgrade report\"\n    locked = {\"rich\": \"13.7.1\", \"idna\": \"3.7\", \"click\": \"8.1.7\", \"attrs\": \"23.2.0\"}\n    available = {\"rich\": [\"13.7.1\", \"14.0.0\", \"13.9.4\"], \"idna\": [\"3.6\", \"3.7\"], \"click\": [\"8.1.8\", \"8.1.7\"], \"attrs\": [\"24.1\", \"23.2.0\"]}\n    assert outdated(locked, available) == [\n        (\"attrs\", \"23.2.0\", \"24.1\", \"major\"),\n        (\"rich\", \"13.7.1\", \"14.0.0\", \"major\"),\n        (\"click\", \"8.1.7\", \"8.1.8\", \"patch\"),\n    ]\n    assert outdated({\"a\": \"1.0\"}, {\"a\": [\"1.0\"]}) == []")

### Exercise: Why greedy isn't enough

Write `greedy_resolve(roots, index)`: `index` maps each package to a dict of version to its requirements (a dict of package to specifier string, using `matches`, given). Starting from `roots` (a dict of package to specifier), repeatedly take the next package needing a version (first-in, first-out), pick its **newest** version satisfying every specifier collected so far for it, and add its requirements. **Never** go back: if a package already has a version that a new requirement rejects, or a package has no matching version, raise `ValueError(f"conflict on {name}")`. Return the resolution. (Today's challenge fixes this with backtracking.)

In [ ]:
from collections import deque

def matches(version, specifier):
    def t(v):
        return tuple(int(p) for p in v.split("."))
    for clause in filter(None, (c.strip() for c in specifier.split(","))):
        op = next(o for o in ("<=", ">=", "==", "!=", "<", ">") if clause.startswith(o))
        a, b = t(version), t(clause[len(op):])
        if not {"<=": a <= b, ">=": a >= b, "==": a == b, "!=": a != b, "<": a < b, ">": a > b}[op]:
            return False
    return True

def greedy_resolve(roots, index):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_greedy():\n    \"newest first, never back\"\n    index = {\n        \"web\": {\"2.0\": {\"http\": \">=2\"}, \"1.0\": {\"http\": \">=1\"}},\n        \"report\": {\"1.0\": {\"http\": \"<2\"}},\n        \"http\": {\"2.1\": {}, \"2.0\": {}, \"1.5\": {}},\n        \"cli\": {\"3.0\": {\"web\": \">=1\"}},\n    }\n    assert greedy_resolve({\"web\": \">=1\", \"cli\": \"\"}, index) == {\"web\": \"2.0\", \"cli\": \"3.0\", \"http\": \"2.1\"}\n    assert greedy_resolve({\"report\": \"\", \"web\": \"<2\"}, index) == {\"report\": \"1.0\", \"web\": \"1.0\", \"http\": \"1.5\"}\n    try:\n        greedy_resolve({\"web\": \"\", \"report\": \"\"}, index)\n        assert False\n    except ValueError as e:\n        assert str(e) == \"conflict on http\"")

### Bug hunt: A specifier checked as text

The check compares versions as strings, so `"1.10"` looks smaller than `"1.9"` and the program wrongly refuses the newer version, printing `False`. Compare tuples of numbers so this prints `True`.

In [ ]:
def at_least(version, minimum):
    return version >= minimum

print(at_least("1.10", "1.9"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"numbers, not text\"\n    expect_output(\"True\")")

### Bug hunt: A lock file that forgot the transitive dependencies

The lock lists only the project's **direct** dependencies, so `http` and `idna` (needed by them) would be installed at whatever version happens to be newest, and this prints `2 packages locked`. Lock everything reachable from the direct dependencies, so it prints `4 packages locked`.

In [ ]:
requires = {"web": ["http"], "report": ["http", "idna"], "http": ["idna"], "idna": []}
direct = ["web", "report"]

locked = sorted(set(direct))
print(len(locked), "packages locked")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"everything that gets installed\"\n    expect_output(\"4 packages locked\")")

### Bug hunt: A wildcard taken literally

`==2.*` should accept any version beginning with 2, but the check compares the whole text, so `2.4.1` is refused and this prints `[]`. Treat a trailing `.*` as a prefix match on the version's numbers, so this prints `['2.0', '2.4.1']`.

In [ ]:
def matches_exact(version, target):
    return version == target

versions = ["1.9", "2.0", "2.4.1", "3.0"]
print([v for v in versions if matches_exact(v, "2.*")])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"prefix matching\"\n    expect_output(\"['2.0', '2.4.1']\")")

### Puzzle: Locking an application

Arrange the commands that lock an application's dependencies with hashes and install exactly those, using pip-tools. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the two clauses that "~=2.3.1" stands for, joined with ", "
q = ___   # True or False: a library should pin its dependencies with ==
r = ___   # True or False: "1.4.0" matches "==1.4"
s = ___   # the pip option that refuses any file whose hash isn't in the lock file
t = ___   # what a resolver does when a package has no version left that fits: "backtrack" or "pick the newest anyway"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \">=2.3.1, ==2.3.*\" and q is False and r is True, \"p, q and r\"\n    assert s == \"--require-hashes\" and t == \"backtrack\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a backtracking resolver

Write `resolve(roots, index)`: like `greedy_resolve`, but with **backtracking**. Packages are decided in first-in, first-out order of when they were first required; for each, try versions **newest first** that satisfy every specifier collected so far; choosing a version adds its requirements, which must also hold for packages already chosen. If no version of a package works, undo the most recent choice and try the next version of **that** package; if the search is exhausted, raise `ValueError` with the message `f"no versions of {name} satisfy: {specs}"`, where `name` is the **first** package that ran out of versions in the whole search and `specs` its collected specifiers at that moment, joined with `"; "`. Return the resolution. (`matches` is given.)

In [ ]:
def matches(version, specifier):
    def t(v):
        return tuple(int(p) for p in v.split("."))
    for clause in filter(None, (c.strip() for c in specifier.split(","))):
        op = next(o for o in ("<=", ">=", "==", "!=", "<", ">") if clause.startswith(o))
        a, b = t(version), t(clause[len(op):])
        if not {"<=": a <= b, ">=": a >= b, "==": a == b, "!=": a != b, "<": a < b, ">": a > b}[op]:
            return False
    return True

def resolve(roots, index):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_resolve():\n    \"backtracking to a solution, and explaining a conflict\"\n    index = {\n        \"web\": {\"2.0\": {\"http\": \">=2\"}, \"1.0\": {\"http\": \">=1\"}},\n        \"report\": {\"1.0\": {\"http\": \"<2\"}},\n        \"http\": {\"2.1\": {}, \"2.0\": {}, \"1.5\": {}},\n        \"cli\": {\"3.0\": {\"web\": \">=1\"}},\n    }\n    assert resolve({\"web\": \"\", \"report\": \"\"}, index) == {\"web\": \"1.0\", \"report\": \"1.0\", \"http\": \"1.5\"}\n    assert resolve({\"web\": \">=1\", \"cli\": \"\"}, index) == {\"web\": \"2.0\", \"cli\": \"3.0\", \"http\": \"2.1\"}\n    deep = {\n        \"a\": {\"3\": {\"b\": \">=2\"}, \"2\": {\"b\": \">=1\"}, \"1\": {}},\n        \"b\": {\"2\": {\"c\": \"==2\"}, \"1\": {\"c\": \"==1\"}},\n        \"c\": {\"2\": {}, \"1\": {}},\n        \"d\": {\"1\": {\"c\": \"<2\"}},\n    }\n    assert resolve({\"a\": \"\", \"d\": \"\"}, deep) == {\"a\": \"2\", \"d\": \"1\", \"b\": \"1\", \"c\": \"1\"}\n    try:\n        resolve({\"web\": \">=2\", \"report\": \"\"}, index)\n        assert False\n    except ValueError as e:\n        assert str(e) == \"no versions of http satisfy: >=2; <2\", str(e)")

## Warm-up

Your project needs `web` and `report`. The newest `web` needs `http` 2 or later; `report` needs `http` before 2. No version of `http` satisfies both, so the newest `web` can't be used. Going back to an older `web` makes everything fit. Finding that combination automatically is the **resolver's** job, and today you'll write one.

## Specifiers

A requirement such as `rich>=13,<15` (Day 2) has a **specifier**: comma-separated clauses, all of which must hold.

| Clause | Means |
|---|---|
| `==1.4.2` | exactly that version (trailing zeros ignored: `==1.4` is the same as `==1.4.0`) |
| `==1.4.*` | any version starting with 1.4 (a **prefix** match) |
| `!=1.4.2`, `!=1.4.*` | anything except |
| `<`, `<=`, `>`, `>=` | ordered comparisons |
| `~=1.4.2` | **compatible release**: `>=1.4.2, ==1.4.*`. With two numbers, `~=1.4` means `>=1.4, ==1.*` |
| `===foo` | arbitrary string equality (rare; avoid) |

`~=` reads as "this version or a later one that SemVer says is compatible": `~=2.3` accepts 2.3, 2.4 and 2.99 but not 3.0. Pre-releases are excluded unless a clause names one. Compare versions as parsed values (Day 4), never as strings.

## Ranges, pins and lock files

Two kinds of project need opposite things:

- A **library** (something others install into their own projects) should declare the **widest ranges** it really supports: `rich>=13`, perhaps `<15` if a known future version breaks it. A library that **pins** (`rich==13.7.1`) forces that exact version on every project that uses it, and two such libraries can make each other uninstallable.
- An **application** (something you deploy: a web service, a command-line tool you run, a data pipeline) should be **reproducible**: the same versions on every machine and every deploy, so it can't break because some dependency released something overnight. It keeps its loose requirements in `pyproject.toml` **and** a **lock file** of the exact versions of **everything**, transitive dependencies included.

A lock file records, for every package, the exact version and the **hashes** of the files allowed, so that a compromised or swapped file is refused: `pip install --require-hashes -r requirements.txt`. Common forms: a `requirements.txt` generated by **pip-tools** (`pip-compile`), `uv.lock` from **uv**, `poetry.lock` and `pdm.lock`. `pip freeze` lists what's installed, but without hashes and without telling direct from transitive dependencies. Commit the lock file; regenerate it deliberately to upgrade.

**Dependency groups.** Tools you need while developing (pytest, a linter) aren't dependencies of the package. Put them in an extra (`[project.optional-dependencies] dev = [...]`), or in the newer `[dependency-groups]` table (PEP 735), which isn't published with the package at all.

## Resolving

Given the requirements, the **resolver** chooses one version of every package such that every requirement of every chosen version holds. The approach pip (and most others) use is **backtracking search**, like the backtracking of Week 52:

1. Keep a set of requirements still to satisfy. Pick a package that needs a version.
2. Try its candidate versions, **newest first**, skipping any that break a requirement already collected.
3. Choosing a version adds **its** requirements. Continue with the next package.
4. If some package has no candidate left, **backtrack**: undo the most recent choice and try that package's next-newest version.
5. If every choice has been exhausted, the requirements **conflict**, and a good resolver says which requirements clashed.

Resolution can take a long time when many versions must be tried (and a resolver may download each candidate's metadata to see its requirements), which is one more reason lock files exist: an application resolves once, deliberately, and installs the result.

## Keeping up to date

Locked versions don't stay safe forever: dependencies fix bugs and **security vulnerabilities** in new releases. Good habits:

- **Upgrade regularly and in small steps**, with the tests running: one dependency at a time when something breaks. Tools such as **Dependabot** and **Renovate** open pull requests with upgrades for you.
- **Audit for known vulnerabilities**: `pip-audit` checks your installed (or locked) packages against public advisory databases (Week 79).
- **Read the changelog** before a major upgrade (Day 4 is why that works).
- **Keep dependencies few.** Each one is code you didn't write, can't fully review and must keep updated. A few lines of your own can beat a dependency for a small job; a well-maintained library beats hundreds of lines of yours for a hard one.

## Practice

## On your own computer

Try two ways of locking a project you've made this week:

```bash
pip install pip-tools && pip-compile --generate-hashes pyproject.toml
```

```bash
uv lock && uv tree
```

```bash
pip install pip-audit && pip-audit
```

The first writes a `requirements.txt` like today's lock file; the second writes `uv.lock` and prints the dependency tree (with `(*)` where uv elides a repeated subtree); the third checks what's installed against known vulnerabilities.

## Quiz

## Daily challenge

## Before you go

Run `pip list --outdated` in an environment of yours (on your own computer) and decide, for each line, whether it's a patch, minor or major upgrade and what you'd check first. Tomorrow's Forge puts the week together: you'll build a small packaging tool that checks a project, works out its version, and builds its wheel.